In [1]:
import pandas as pd
import numpy as np
from pulp import LpProblem, LpVariable, lpSum, LpMaximize, LpBinary, PULP_CBC_CMD

In [52]:
# === CONFIGURATION ===

CSV_PATH = 'items.csv'
weights = np.array([1, 120000, 3.33, 30000, 34, 0])   # [HP, CDS, DMG, AS, Armor, MR]
stat_cols = ['HP', 'CDS', 'DMG', 'AS', 'Armor', 'MR']
item_limit = 6

# Custom constraint functions (for illustration)
def constraint_at_least_one_with_cds(x_vars, stats):
    # At least 1 item with cooldown speed > 0
    return lpSum([x_vars[i] for i in range(len(x_vars)) if stats[i, 1] > 0]) >= 1

def constraint_minimum_hp(x_vars, stats):
    # At least 30,000 total HP
    return lpSum([stats[i, 0] * x_vars[i] for i in range(len(x_vars))]) >= 30000

custom_constraints = [
    constraint_at_least_one_with_cds,
    # constraint_minimum_hp,
]

In [39]:
# === DATA LOADING ===
df = pd.read_csv(CSV_PATH).fillna(0)
stats = df[stat_cols].values
num_items = df.shape[0]
item_names = df['Item'].tolist()
soul_crystal_idx = [i for i, n in enumerate(item_names) if n.strip().lower() == "soul crystal"][0]

In [40]:
# === OPTIMIZATION FUNCTION ===
def optimize_items(force_soul_crystal=None, custom_constraints=[]):
    prob = LpProblem("Best_Item_Combo", LpMaximize)
    x_vars = [LpVariable(f"x_{i}", 0, 1, LpBinary) for i in range(num_items)]
    # Objective
    total_stats = [lpSum([stats[i, j] * x_vars[i] for i in range(num_items)]) for j in range(len(stat_cols))]
    prob += lpSum([weights[j] * total_stats[j] for j in range(len(weights))])
    # Constraints
    prob += lpSum(x_vars) == item_limit
    # Soul Crystal inclusion/exclusion
    if force_soul_crystal is not None:
        if force_soul_crystal:
            prob += x_vars[soul_crystal_idx] == 1
        else:
            prob += x_vars[soul_crystal_idx] == 0
    # Custom constraints
    for fn in custom_constraints:
        prob += fn(x_vars, stats)
    prob.solve(PULP_CBC_CMD(msg=0))
    chosen = [i for i in range(num_items) if x_vars[i].varValue > 0.5]
    chosen_items = df.iloc[chosen][['Item'] + stat_cols]
    totals = df.iloc[chosen][stat_cols].sum()
    return chosen_items, totals

In [53]:
# === RUN BUILDS ===
for force_sc, label in [(True, "WITH Soul Crystal"), (False, "WITHOUT Soul Crystal")]:
    items, totals = optimize_items(force_soul_crystal=force_sc, custom_constraints=custom_constraints)
    print(f"\n=== Build {label} ===")
    print(items)
    print("\nTotal Stats:\n", totals)


=== Build WITH Soul Crystal ===
                        Item       HP   CDS     DMG    AS  Armor    MR
0            Plated Gauntlet  12000.0  0.00   600.0  0.20   75.0  0.00
2            Summoning Codex   8500.0  0.05   425.0  0.00   35.0  0.00
4               Soul Crystal   7150.0  0.00     0.0  0.00    0.0  0.25
5   Blade of the Ruined King   7000.0  0.00  3600.0  0.00    0.0  0.00
6                Hand of God   6500.0  0.00   325.0  0.15   25.0  0.00
13                    Tremor      0.0  0.10     0.0  0.00  150.0  0.00

Total Stats:
 HP       41150.00
CDS          0.15
DMG       4950.00
AS           0.35
Armor      285.00
MR           0.25
dtype: float64

=== Build WITHOUT Soul Crystal ===
                        Item       HP   CDS     DMG    AS  Armor   MR
0            Plated Gauntlet  12000.0  0.00   600.0  0.20   75.0  0.0
2            Summoning Codex   8500.0  0.05   425.0  0.00   35.0  0.0
5   Blade of the Ruined King   7000.0  0.00  3600.0  0.00    0.0  0.0
6               